In [ ]:
import hashlib
import json
import time

class Block:
    def __init__(self, index, previous_hash, data):
        self.index = index
        self.timestamp = time.time()
        self.data = data  # Contains letters (A, B, C)
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash
        }, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

    def to_dict(self):
        return {
            "index": self.index,
            "timestamp (UET)": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "hash": self.hash
        }

previous_hash = "0" * 64  
block_data = "A, B, C"  
new_block = Block(index=0, previous_hash=previous_hash, data=block_data)

blockchain = [new_block.to_dict()]

import pprint
pprint.pprint(blockchain)

Alright first block created. What a big brain move. Now time to make an actual blockchain and validate it works.

In [ ]:
import hashlib
import json
import time

class Block:
    def __init__(self, index, previous_hash, data):
        self.index = index
        self.timestamp = time.time()
        self.data = data  # Contains letters (A, B, C)
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash
        }, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

    def to_dict(self):
        return {
            "index": self.index,
            "timestamp (UET)": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "hash": self.hash
        }


class Blockchain:
    def __init__(self):
        self.chain = []
        self.create_genesis_block()

    def create_genesis_block(self):
        """Creates the first block in the blockchain."""
        genesis_block = Block(index=0, previous_hash="0" * 64, data="Genesis Block")
        self.chain.append(genesis_block)

    def add_block(self, data):
        """Adds a new block to the blockchain."""
        previous_block = self.chain[-1]
        new_block = Block(index=previous_block.index + 1, previous_hash=previous_block.hash, data=data)
        self.chain.append(new_block)

    def is_valid_chain(self):
        """Validates the blockchain by checking hash consistency."""
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i - 1]

            # Validate hash linkage
            if current_block.previous_hash != previous_block.hash:
                return False

            # Validate current block hash
            if current_block.hash != current_block.calculate_hash():
                return False

        return True

    def display_chain(self):
        """Displays the blockchain in a readable format."""
        blockchain_data = [block.to_dict() for block in self.chain]
        import pprint
        pprint.pprint(blockchain_data)


# Example Usage
blockchain = Blockchain()
blockchain.add_block("A, B, C")
blockchain.add_block("D, E, F")
blockchain.add_block("G, H, I")

blockchain.display_chain()

# Validate Blockchain
print("Blockchain is valid:", blockchain.is_valid_chain())


Alright easy enough. Now to Parse a blockchain. Will figure this out and try to implement.

In [ ]:
for block in blockchain.chain:
    print(f"Block {block.index}: {block.hash}")


In [ ]:
search_letter = "B"
matching_blocks = [block.to_dict() for block in blockchain.chain if search_letter in block.data]

import pprint
pprint.pprint(matching_blocks)


In [ ]:
blockchain_json = json.dumps([block.to_dict() for block in blockchain.chain], indent=4)
print(blockchain_json)


In [ ]:
blockchain_data = json.loads(blockchain_json)
for block in blockchain_data:
    print(block["index"], block["hash"])


In [ ]:
def validate_blockchain(blockchain):
    for i in range(1, len(blockchain.chain)):
        if blockchain.chain[i].previous_hash != blockchain.chain[i - 1].hash:
            return False
    return True

print("Is blockchain valid?", validate_blockchain(blockchain))


Alright now that we have successfully parsed our blockchain, I will start with using sqlite3 and see if I can actually store my blockchain data into a SQL Servser style environment

In [ ]:
!pip install psycopg2

In [ ]:
import hashlib
import json
import time
import sqlite3

class Block:
    def __init__(self, index, previous_hash, data):
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash
        }, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

    def to_dict(self):
        return {
            "block_index": self.index,  # Changed index to block_index
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "hash": self.hash
        }


class Blockchain:
    def __init__(self):
        self.chain = []
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis_block = Block(index=0, previous_hash="0" * 64, data="Genesis Block")
        self.chain.append(genesis_block)

    def add_block(self, data):
        previous_block = self.chain[-1]
        new_block = Block(index=previous_block.index + 1, previous_hash=previous_block.hash, data=data)
        self.chain.append(new_block)

    def display_chain(self):
        for block in self.chain:
            print(f"Index: {block.index}, Data: {block.data}, Hash: {block.hash}, Previous Hash: {block.previous_hash}")


# Database Functions
def create_database():
    """Creates a SQLite database and a table for storing blockchain data."""
    conn = sqlite3.connect("blockchain.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS blocks (
            block_index INTEGER PRIMARY KEY,
            timestamp REAL,
            data TEXT,
            previous_hash TEXT,
            hash TEXT UNIQUE
        )
    """)
    conn.commit()
    conn.close()


def store_block(block):
    """Stores a block in the database."""
    with sqlite3.connect("blockchain.db") as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO blocks (block_index, timestamp, data, previous_hash, hash)
            VALUES (?, ?, ?, ?, ?)
        """, (block.index, block.timestamp, block.data, block.previous_hash, block.hash))
        conn.commit()  # Ensure changes are committed



def display_database():
    """Displays all blocks stored in the SQL database."""
    with sqlite3.connect("blockchain.db") as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM blocks ORDER BY block_index ASC")
        blocks = cursor.fetchall()

    for block in blocks:
        print(f"Index: {block[0]}, Data: {block[2]}, Hash: {block[4]}, Previous Hash: {block[3]}")


# Example Usage
create_database()
blockchain = Blockchain()
blockchain.add_block("A, B, C")
blockchain.add_block("D, E, F")
blockchain.add_block("G, H, I")

# Store blockchain in SQL
for block in blockchain.chain:
    store_block(block)

# Display stored blockchain data from database
print("\nBlockchain from Database:")
display_database()
